In [ ]:
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/isamagana/puremacro.git'
REPO_DIR = Path('/content/puremacro')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
DATA_DIR = REPO_DIR / 'curso_data'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from puremacro.fetch import qna_countries, qna_panel, qna_meta
from puremacro.sa import residual_seasonality_F, deseasonalize_x13
from puremacro.labor_share import gollin_adjusted_ls
from puremacro.data import hp_filter
from puremacro.cycles import hamilton_filter

D_ANUAL = {'inv_equip': .13, 'inv_struct': .030, 'inv_dwell': .011,
           'inv_ipp': .20, 'cons_dur': .21, 'inv_gov': .04}
D_Q = {k: 1-(1-v)**.25 for k,v in D_ANUAL.items()}
R_Q = (1.04)**.25-1


In [ ]:
#Parte 1

paises = qna_countries()
panel_descargado = qna_panel(paises, start='1995', assets=True, durability=True,
                             income=True, labor=True, real=True, sa='x13')
if panel_descargado.empty:
    raise RuntimeError('La descarga trimestral de la OCDE no devolvió datos. Reintenta en Colab.')
meta = qna_meta(panel_descargado)
panel = panel_descargado.reset_index()
print('Países descargados:', panel.code.nunique())
COLS = ['gdp','cons_hh','cons_gov','inv','exports','imports','cons_dur',
        'inv_equip','inv_struct','inv_dwell','inv_ipp','comp_emp',
        'surplus_mixed','taxes_prod_imp_net','gdp_income',
        'emp_employees','emp_selfemp']
DEFLS = ['inv_defl','cons_dur_defl','inv_equip_defl',
         'inv_struct_defl','inv_dwell_defl','inv_ipp_defl']
faltan = set(COLS+DEFLS)-set(panel)
if faltan:
    raise ValueError(f'Columnas indispensables faltantes: {sorted(faltan)}')
print('Metadatos de base de precios:')
print(meta[[c for c in ['code','price_base','price_ref_year'] if c in meta]].drop_duplicates().head())
print('VERIFICA CON LA DOCUMENTACIÓN DEL PANEL: las columnas de flujo deben ser a precios corrientes.')

ruta_ig = DATA_DIR/'inv_gov.csv'
PUBLICO = ruta_ig.exists()
if PUBLICO:
    ig = leer('inv_gov.csv')
    if not {'code','date','inv_gov'}.issubset(ig):
        raise ValueError('inv_gov.csv requiere code,date,inv_gov (inversión pública trimestral nominal).')
    panel = panel.merge(ig[['code','date','inv_gov']],on=['code','date'],how='left',validate='one_to_one')
    COLS.append('inv_gov')
else:
    print('SIN inv_gov.csv TRIMESTRAL: las columnas de capital público y la parte completa de la tarea quedarán pendientes.')

panel = panel.sort_values(['code','date']).reset_index(drop=True)
print('Panel:', len(panel), 'observaciones y', panel.code.nunique(), 'países.')

def corregir_estacionalidad(df, columnas, alpha=.05):
    df = df.copy()
    auditoria = []
    for col in columnas:
        if col not in df: continue
        for pais, indices in df.groupby('code',sort=False).groups.items():
            sub = df.loc[indices, ['code','date',col]].dropna().sort_values('date')
            if len(sub)<24: continue
            try:
                f,p,*_ = residual_seasonality_F(sub.set_index('date')[col],freq='Q')
            except Exception as e:
                auditoria.append((pais,col,np.nan,'prueba falló: '+str(e)))
                continue
            motor='sin ajuste'
            if np.isfinite(p) and p<alpha:
                motores={}
                ajustada = deseasonalize_x13(sub, value_col=col, by='code',
                         date_col='date',freq='Q',engines=motores)
                motor=str(motores)
                if 'stl' in motor.lower():
                    raise RuntimeError(f'El motor STL no es admisible para {pais} {col}')
                if ajustada.isna().any():
                    raise RuntimeError(f'El ajuste perdió observaciones: {pais} {col}')
                df.loc[sub.index,col] = ajustada.to_numpy()
                f,p,*_ = residual_seasonality_F(
                    df.loc[sub.index].sort_values('date').set_index('date')[col],freq='Q')
                if np.isfinite(p) and p<alpha:
                    raise RuntimeError(f'Persiste estacionalidad en {pais} {col}: p={p}')
            auditoria.append((pais,col,p,motor))
    tabla = pd.DataFrame(auditoria,columns=['code','variable','p_final','motor'])
    return df,tabla

panel, auditoria_sa = corregir_estacionalidad(panel,COLS+DEFLS)
print('Pruebas:',len(auditoria_sa),'reajustes:',(auditoria_sa.motor!='sin ajuste').sum(),
      'errores:',auditoria_sa.motor.str.startswith('prueba falló').sum())
print(auditoria_sa[auditoria_sa.motor!='sin ajuste'].to_string(index=False))
if auditoria_sa.motor.str.startswith('prueba falló').any():
    raise RuntimeError('Revisa las series cuya prueba estacional falló.')
panel = panel.set_index(['code','date']).sort_index()


In [ ]:
#Parte 2

DEF_BASE = 100.0

def pim_real(pais, gasto_col, defl_col, delta, minimo=8):
    x = pais[[gasto_col,defl_col]].copy()
    out = pd.DataFrame(index=x.index,columns=['K_real','serv_nom'],dtype=float)
    ok = x[gasto_col].gt(0) & x[defl_col].gt(0)
    ordinal = x.index.to_period('Q').astype(int)
    cortes = ok.ne(ok.shift(fill_value=False)) | (pd.Series(ordinal,index=x.index).diff()!=1)
    for _, bloque in x.groupby(cortes.cumsum()):
        idx=bloque.index
        if len(idx)<minimo or not ok.loc[idx].all(): continue
        inversion = bloque[gasto_col].to_numpy(dtype=float)/(bloque[defl_col].to_numpy(dtype=float)/DEF_BASE)
        d0 = inversion[0]; d7=inversion[7]
        g = max((d7/d0)**(1/7)-1, -0.5*delta)
        k=np.empty(len(inversion)); k[0]=d0/(g+delta)
        for t in range(1,len(k)):
            k[t]=(1-delta)*k[t-1]+inversion[t-1]
        out.loc[idx,'K_real']=k
        out.loc[idx,'serv_nom']=(R_Q+delta)*k*(bloque[defl_col].to_numpy(dtype=float)/DEF_BASE)
    return out

def hacer_capital(df, col, defl, delta, nombre):
    partes=[]
    for pais, sub in df.groupby(level='code',sort=False):
        r=pim_real(sub.droplevel('code'),col,defl,delta)
        r.columns=[f'K_{nombre}_real',f'S_{nombre}_nom']
        r['code']=pais
        partes.append(r.reset_index().set_index(['code','date']))
    return pd.concat(partes).reindex(df.index)

panel['NX']=panel['exports']-panel['imports']
panel['A']=panel['cons_hh']+panel['cons_gov']+panel['inv']
panel['NX_Y']=panel['NX']/panel['gdp']
panel['error_gasto_publicado']=(panel['gdp']-panel['NX'])-panel['A']
print('NX/Y: media y desviación por país')
print(panel.groupby(level='code')['NX_Y'].agg(['mean','std']).round(3))
print('Residual de absorción vs. Y-NX:',panel.error_gasto_publicado.abs().max())

panel=panel.join(hacer_capital(panel,'cons_dur','cons_dur_defl',D_Q['cons_dur'],'d'))
activos=['inv_equip','inv_struct','inv_dwell','inv_ipp']
for col in activos:
    panel=panel.join(hacer_capital(panel,col,col+'_defl',D_Q[col],col))
capital_real=[f'K_{col}_real' for col in activos]
panel['Kp_real']=panel[capital_real].sum(axis=1,min_count=4)
if PUBLICO:
    panel=panel.join(hacer_capital(panel,'inv_gov','inv_defl',D_Q['inv_gov'],'g'))
    panel['Rg_nom']=R_Q*panel['K_g_real']*(panel['inv_defl']/DEF_BASE)
else:
    panel['Rg_nom']=np.nan

panel['C_sin_g']=panel.cons_hh-panel.cons_dur+panel.cons_gov+panel.S_d_nom
panel['I_star']=panel.inv+panel.cons_dur
panel['Y_sin_g']=panel.A+panel.S_d_nom
panel['C_estricta']=panel.C_sin_g-panel.cons_gov
panel['Y_estricta']=panel.Y_sin_g-panel.cons_gov
panel['robustez_Cg_Y']=panel.cons_gov/panel.Y_sin_g
if PUBLICO:
    panel['C_star']=panel.C_sin_g+panel.Rg_nom
    panel['Y_star']=panel.Y_sin_g+panel.Rg_nom
else:
    panel['C_star']=np.nan
    panel['Y_star']=np.nan
Ycol='Y_star' if PUBLICO else 'Y_sin_g'
Ccol='C_star' if PUBLICO else 'C_sin_g'
valido=panel[[Ycol,Ccol,'I_star']].notna().all(axis=1)
error=panel.loc[valido,Ycol]-panel.loc[valido,Ccol]-panel.loc[valido,'I_star']
print('Identidad',Ycol,'= C + I; error máximo:',error.abs().max(),'observaciones:',len(error))

panel['Cd_Ch']=panel.cons_dur/panel.cons_hh
panel['Sd_Y']=panel.S_d_nom/panel[Ycol]
panel['Rg_Y']=panel.Rg_nom/panel.Y_star if PUBLICO else np.nan
panel['imputado_C']=(panel.S_d_nom+panel.Rg_nom)/panel.C_star if PUBLICO else panel.S_d_nom/panel.C_sin_g
print('Proporciones: medias de cocientes trimestrales (fracción; x100 para %)')
print(panel.groupby(level='code')[['Cd_Ch','Sd_Y','Rg_Y','imputado_C']].mean().round(4))
print('Robustez (a vs b), diferencia porcentual sobre Y ampliado:')
print((100*panel.groupby(level='code')['robustez_Cg_Y'].mean()).round(2))


In [ ]:
#Parte 3

df_labor=panel.reset_index()[['code','date','comp_emp','gdp_income','emp_employees','emp_selfemp']].rename(columns={
    'comp_emp':'compensation_employees','gdp_income':'value_added',
    'emp_employees':'employment_employees','emp_selfemp':'employment_self'})
df_labor = df_labor.dropna(subset=['compensation_employees','value_added',
     'employment_employees','employment_self'])
df_labor = df_labor[(df_labor.value_added>0)&(df_labor.employment_employees>0)]
gollin=gollin_adjusted_ls(df_labor).set_index(['code','date'])['ls_gollin']
panel['ls_gollin']=gollin.reindex(panel.index)
panel['Yinc_publicado']=panel[['comp_emp','surplus_mixed','taxes_prod_imp_net']].sum(axis=1,min_count=3)
panel['Yinc_cerrado']=panel.Yinc_publicado-panel.NX
panel['Yinc_star']=panel.Yinc_cerrado+panel.S_d_nom+(panel.Rg_nom if PUBLICO else 0)
panel['L_cerrado']=panel.ls_gollin*panel.Yinc_cerrado
panel['ls_transformada']=panel.L_cerrado/panel.Yinc_star
panel['caida_pp']=100*(panel.ls_gollin-panel.ls_transformada)
panel['brecha_pct']=100*(panel[Ycol]-panel.Yinc_star)/panel[Ycol]
print('Participación laboral y caída en puntos porcentuales:')
print(panel.groupby(level='code')[['ls_gollin','ls_transformada','caida_pp']].mean().dropna().round(3))
print('Brecha gasto - ingreso, % del producto ajustado; media y desviación:')
print(panel.groupby(level='code')['brecha_pct'].agg(['mean','std']).dropna().round(2))
print('Comprueba además el residual original Y del gasto - Y del ingreso:')
print((panel.gdp-panel.Yinc_publicado).groupby(level='code').mean().dropna().round(2))


In [ ]:
#Parte 4

def valor_actual_capital(p, cols):
    return sum(p[f'K_{c}_real']*p[c+'_defl']/DEF_BASE for c in cols)

panel['Kp_nom']=valor_actual_capital(panel,activos)
panel['Kd_nom']=panel.K_d_real*panel.cons_dur_defl/DEF_BASE
panel['Kg_nom']=panel.K_g_real*panel.inv_defl/DEF_BASE if PUBLICO else np.nan
panel['K_Y_publicado']=panel.Kp_nom/panel.gdp
panel['K_Y_parcial']=(panel.Kp_nom+panel.Kd_nom)/panel.Y_sin_g
panel['K_Y_completo']=(panel.Kp_nom+panel.Kd_nom+panel.Kg_nom)/panel.Y_star if PUBLICO else np.nan
kcol='K_Y_completo' if PUBLICO else 'K_Y_parcial'
comparables=panel[['K_Y_publicado',kcol]].dropna()
print('Razones promedio en trimestres comunes; capital completo' if PUBLICO else 'Razones promedio PARCIALES en trimestres comunes')
print(comparables.groupby(level='code').mean().round(3))
fig,ax=plt.subplots(figsize=(9,5))
for pais in ['USA','DEU','MEX','JPN','ESP']:
    if pais in comparables.index.get_level_values('code'):
        z=comparables.loc[pais]
        ax.plot(z.index,z[kcol],label=pais)
ax.set(xlabel='Fecha',ylabel='Acervo / producto trimestral',title='Capital/producto: serie '+('completa' if PUBLICO else 'parcial'))
ax.legend();ax.grid(alpha=.3);plt.show()

paises_fig = [p for p in ['USA','DEU','MEX','JPN','ESP']
              if p in comparables.index.get_level_values('code')]
fig, axes = plt.subplots(len(paises_fig), 1, figsize=(10, 2.5*len(paises_fig)), sharex=True)
if len(paises_fig)==1: axes=[axes]
for ax, pais in zip(axes,paises_fig):
    z=comparables.loc[pais]
    ax.plot(z.index,z['K_Y_publicado'],label='K privado / PIB publicado',alpha=.85)
    ax.plot(z.index,z[kcol],label='K ampliado / producto ajustado',alpha=.85)
    ax.set_ylabel(pais)
    ax.grid(alpha=.25)
axes[0].legend(loc='best')
axes[-1].set_xlabel('Fecha')
fig.suptitle('Razón capital/producto, trimestres comunes por país')
fig.tight_layout()
plt.show()


In [ ]:
#Parte 5

def momentos_ciclicos(p):
    z=pd.DataFrame({'Y':p.gdp,'C':p.cons_hh+p.cons_gov,'I':p.inv,
                    'Y*':p[Ycol],'C*':p[Ccol],'I*':p.I_star}).dropna()
    z=z[(z>0).all(axis=1)]
    if len(z)<32:return []
    ordinal=pd.Series(z.index.to_period('Q').astype(int),index=z.index)
    grupos=(ordinal.diff()!=1).cumsum()
    z=max((g for _,g in z.groupby(grupos)),key=len)
    if len(z)<32:return []
    logs=100*np.log(z)
    resultados=[]
    for filtro in ['HP','Hamilton']:
        c=pd.DataFrame(index=z.index)
        for col in logs:
            cyc,_=(hp_filter(logs[col],lamb=1600) if filtro=='HP' else
                   hamilton_filter(logs[col],h=8,p=4))
            c[col]=pd.Series(cyc,index=z.index)
        c=c.dropna()
        if len(c)<15:continue
        for tipo,Y,cols in [('publicada','Y',['Y','C','I']),
                            ('transformada','Y*',['Y*','C*','I*'])]:
            sy=c[Y].std()
            for col in cols:
                sx=c[col].std()
                resultados.append({'filtro':filtro,'tipo':tipo,'variable':col,
                  'sigma_y':sy,'sigma_x':sx,'sigma_rel':sx/sy,'corr_y':c[col].corr(c[Y]),
                  'n':len(c)})
    return resultados

corto=panel[panel.index.get_level_values('date')<pd.Timestamp('2020-01-01')]
rows=[]
for pais,sub in corto.groupby(level='code'):
    for r in momentos_ciclicos(sub.droplevel('code')):
        rows.append({'code':pais,**r})
if not rows: raise RuntimeError('No hay países con seis series y al menos 32 trimestres completos.')
momentos=pd.DataFrame(rows)
print('Países con comparación:',momentos.code.nunique())
print(momentos.groupby(['filtro','tipo','variable'])[['sigma_y','sigma_rel','corr_y']].mean().round(3))

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex='col')
for i,filtro in enumerate(['HP','Hamilton']):
    z=momentos[momentos.filtro.eq(filtro)]
    promedios=z.groupby(['tipo','variable'])[['sigma_rel','corr_y']].mean()
    etiquetas=['C','C*','I','I*']
    colores=['#337ab7','#d97523','#337ab7','#d97523']
    xs=np.arange(4)
    for j,(medida,titulo) in enumerate([('sigma_rel','Volatilidad relativa'),
                                         ('corr_y','Correlación con el producto')]):
        vals=[promedios.loc[('publicada' if v in ('C','I') else 'transformada',v),medida]
              for v in etiquetas]
        ax=axes[i,j]
        ax.bar(xs,vals,color=colores)
        ax.set_xticks(xs,etiquetas)
        ax.set_title(f'{filtro}: {titulo}')
        ax.grid(axis='y',alpha=.25)
        for k,v in enumerate(vals):ax.text(k,v,f'{v:.2f}',ha='center',va='bottom',fontsize=9)
fig.suptitle('Momentos cíclicos hasta 2019T4 (países comparables)')
fig.tight_layout()
plt.show()

def cobertura(cols,umbral=.9):
    def ok(p):
        base=p.gdp.notna().sum()
        return base>0 and p[cols].notna().all(axis=1).sum()/base>=umbral
    return sum(ok(p) for _,p in panel.groupby(level='code'))
niveles={
 'gasto completo': ['gdp','cons_hh','cons_gov','inv','exports','imports'],
 '+ duraderos': ['gdp','cons_hh','cons_gov','inv','exports','imports','cons_dur','K_d_real','S_d_nom'],
 '+ cuatro activos': ['gdp','cons_hh','cons_gov','inv','exports','imports','cons_dur','K_d_real','S_d_nom','Kp_real'],
 '+ ingreso y empleo': ['gdp','cons_hh','cons_gov','inv','exports','imports','cons_dur','K_d_real','S_d_nom','Kp_real','Yinc_publicado','ls_gollin'],
}
if PUBLICO:
    niveles['+ inversión pública trimestral y Kg'] = niveles['+ cuatro activos']+['inv_gov','K_g_real','Rg_nom']
    niveles['+ lado del ingreso con Kg'] = niveles['+ ingreso y empleo']+['inv_gov','K_g_real','Rg_nom']
    print('La columna Kehoe del enunciado exige además δKg observado; el supuesto delta=.04 no es esa serie.')
tabla_cobertura=pd.Series({k:cobertura(v) for k,v in niveles.items()},name='n_paises')
print(tabla_cobertura.to_string())
if not PUBLICO:
    print('PENDIENTE: comparación completa con Kg y columnas Hall/Kehoe; requiere datos trimestrales apropiados.')
fallos_derivados=[]
for nombre in [Ycol,Ccol,'I_star','Kp_real','K_d_real']:
    for pais,sub in panel.groupby(level='code'):
        z=sub.droplevel('code')[nombre].dropna()
        if len(z)<24: continue
        try:
            _,p,*_=residual_seasonality_F(z,freq='Q')
            if np.isfinite(p) and p<.05: fallos_derivados.append((pais,nombre,p))
        except Exception as e:
            fallos_derivados.append((pais,nombre,str(e)))
print('Problemas estacionales de series derivadas:',fallos_derivados)
if fallos_derivados:
    print('Revisa las entradas y reconstruye todas las variables; NO ajustes Y, C, I por separado.')
print('FIN: no reutilices cifras anteriores; exporta estas tablas solo tras ejecutar todas las celdas en orden.')
